In [1]:
import pandas as pd
import yaml

In [2]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

In [4]:
air_nuts_clean = pd.read_csv(config['data']['cleam']['file10'])
df_life_exp_nuts2_long_clean = pd.read_csv(config['data']['cleam']['file11'], sep=';')
df_mortality_nuts2_long = pd.read_csv(config['data']['cleam']['file12'], sep=';')
water_nuts_clean = pd.read_csv(config['data']['cleam']['file13'])


In [13]:
def df_info(df):
    print("\n    ----- Shape -----")
    print(df.shape)
    #print("\n    ---- df info ---")
    #print(df.info())
    print("\n    ---- Empty Values ---")
    print(df.isna().sum())
    print("\n    ----- Dtypes -----")
    print(df.dtypes)
    #print("\n    ---- Statistics ---")
    return df.describe()

print("\n    ---- AIR ---")
df_info(air_nuts_clean)
print("\n    ---- WATER ---")
df_info(water_nuts_clean)
print("\n    ---- LIFE EXPECTANCY ---")
df_info(df_life_exp_nuts2_long_clean)
print("\n    ---- MORTALITY RATES ---")
df_info(df_mortality_nuts2_long)



    ---- AIR ---

    ----- Shape -----
(366206, 14)

    ---- Empty Values ---
countryname             0
reportingyear           0
eprtr_sectorname        0
facilityname            0
city                    0
longitude               0
latitude                0
targetrelease           0
pollutant               0
releases                0
nuts_id             44345
cntr_code           44345
nuts_name           44345
iso3_code           44345
dtype: int64

    ----- Dtypes -----
countryname          object
reportingyear         int64
eprtr_sectorname     object
facilityname         object
city                 object
longitude           float64
latitude            float64
targetrelease        object
pollutant            object
releases              int64
nuts_id              object
cntr_code            object
nuts_name            object
iso3_code            object
dtype: object

    ---- WATER ---

    ----- Shape -----
(251384, 14)

    ---- Empty Values ---
countryname             0
rep

,year,mortality_rate
count,396006.000000,396006.000000
mean,2017.000000,146.721848
std,3.741662,380.734775
min,2011.000000,0.000000
25%,2014.000000,3.900000
50%,2017.000000,25.260000
75%,2020.000000,118.430000
max,2023.000000,7254.020000


In [14]:
df_life_exp_nuts2_long_clean.head()

,freq,unit,sex,age,geo_time_period,year,life_expectancy
0,A,YR,F,Y_LT1,AL01,2010,NaN
1,A,YR,F,Y_LT1,AL02,2010,NaN
2,A,YR,F,Y_LT1,AL03,2010,NaN
3,A,YR,F,Y_LT1,AT11,2010,"83,9"
4,A,YR,F,Y_LT1,AT12,2010,"83,3"


## AIR + WATER

In [16]:
air_nuts_clean['medium'] = 'air'
water_nuts_clean['medium'] = 'water'
pollution = pd.concat([air_nuts_clean, water_nuts_clean], ignore_index=True)
pollution.head()

,countryname,reportingyear,eprtr_sectorname,facilityname,city,longitude,latitude,targetrelease,pollutant,releases,nuts_id,cntr_code,nuts_name,iso3_code,medium
0,Germany,2013,Energy sector,RWE Power AG-Fabrik Fortuna Nord,Bergheim,6.658050,50.986950,AIR,Sulphur oxides (SOX),824000,DEA2,DE,Köln,DEU,air
1,Sweden,2020,Intensive livestock production and aquaculture,Skåneägg Produktion AB,MÖRARP,12.849550,56.053160,AIR,Ammonia (NH3),16300,SE22,SE,Sydsverige,SWE,air
2,Romania,2013,Waste and wastewater management,SC ECOREC SA,TULCEA,28.769671,45.190459,AIR,Methane (CH4),269000,RO22,RO,Sud-Est,ROU,air
3,Italy,2021,Intensive livestock production and aquaculture,RONCHETRIN 1,GAZZO VERONESE,11.106667,45.123611,AIR,Ammonia (NH3),31920,ITH3,IT,Veneto,ITA,air
4,Czechia,2007,Energy sector,energetika,Štětí,14.380370,50.462296,AIR,Sulphur oxides (SOX),507000,CZ04,CZ,Severozápad,CZE,air


In [18]:
pollution_by_region = (
    pollution
    .groupby(['nuts_id', 'nuts_name', 'cntr_code', 'reportingyear',
              'medium', 'targetrelease', 'pollutant'], dropna=False)
    ['releases']
    .sum()
    .reset_index()
    .rename(columns={'reportingyear': 'year'})
)
pollution_by_region.head()

,nuts_id,nuts_name,cntr_code,year,medium,targetrelease,pollutant,releases
0,AT11,Burgenland,AT,2007,air,AIR,Methane (CH4),1054000
1,AT11,Burgenland,AT,2007,air,AIR,Non-methane volatile organic compounds (NMVOC),432000
2,AT11,Burgenland,AT,2007,water,WATER,Total organic carbon(as total C or COD/3) (TOC),55300
3,AT11,Burgenland,AT,2008,air,AIR,Methane (CH4),933000
4,AT11,Burgenland,AT,2008,air,AIR,Non-methane volatile organic compounds (NMVOC),425000


## HEALTH

In [19]:
df_life_exp_nuts2_long_clean['life_expectancy'] = pd.to_numeric(
    df_life_exp_nuts2_long_clean['life_expectancy'], errors='coerce'
)

life_exp_total = df_life_exp_nuts2_long_clean.loc[
    (df_life_exp_nuts2_long_clean['sex'] == 'T') &
    (df_life_exp_nuts2_long_clean['age'] == 'TOTAL'),
    ['geo_time_period', 'year', 'life_expectancy']
]

mortality_total = df_mortality_nuts2_long.loc[
    (df_mortality_nuts2_long['sex'] == 'T') &
    (df_mortality_nuts2_long['age'] == 'TOTAL'),
    ['geo_time_period', 'year', 'icd10', 'mortality_rate']
]
mortality_total.head()

,geo_time_period,year,icd10,mortality_rate
20019,AT11,2011,A_B,6.17
20020,AT12,2011,A_B,5.88
20021,AT13,2011,A_B,12.05
20022,AT21,2011,A_B,6.75
20023,AT22,2011,A_B,7.74
